# Adverse Discovery: Summary Statistics
Analysis of adversely predicted instances across all datasets

In [53]:
def analyze_dataset(dataset_name, target_col, adverse_file, dataset_sensitive_attrs):
    # Generate comprehensive summary statistics for adversely predicted instances
    
    dataset_path = DATASETS_BASE / f'{dataset_name}_dataset'
    
    print(f"\n{'='*80}")
    print(f"DATASET: {dataset_name.upper()}")
    print(f"{'='*80}\n")
    
    # ===== a) TOTAL INSTANCES =====
    train_df = pd.read_parquet(dataset_path / 'train_cleaned.parquet')
    test_df = pd.read_parquet(dataset_path / 'test_cleaned.parquet')
    total_instances = len(train_df) + len(test_df)
    
    print(f"a) Total Instances in Dataset:")
    print(f"   - Train set: {len(train_df):,} instances")
    print(f"   - Test set:  {len(test_df):,} instances")
    print(f"   - Total:     {total_instances:,} instances")
    
    # ===== b) ADVERSELY PREDICTED INSTANCES =====
    adverse_df = pd.read_csv(dataset_path / adverse_file)
    num_adverse = len(adverse_df)
    adverse_in_test = (adverse_df['original_test_index'] < len(test_df)).sum()
    adverse_pct = (num_adverse / len(test_df)) * 100
    
    print(f"\nb) Adversely Predicted Instances (class 1):")
    print(f"   - Total adverse in test set: {num_adverse} ({adverse_pct:.1f}%)")
    
    # ===== c) TRAIN/TEST SPLIT =====
    print(f"\nc) Train/Test Split Used:")
    print(f"   - Train: {len(train_df) / total_instances * 100:.1f}%")
    print(f"   - Test:  {len(test_df) / total_instances * 100:.1f}%")
    
    # ===== d) MODEL PERFORMANCE =====
    X_test = pd.read_parquet(dataset_path / 'test_cleaned.parquet')
    y_test = X_test[target_col]
    X_test_features = X_test.drop(columns=[target_col])
    
    model_path = dataset_path / 'RF.pkl'
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    
    y_pred = model.predict(X_test_features)
    y_pred_proba = model.predict_proba(X_test_features)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\nd) Model Performance on Test Set:")
    print(f"   - Accuracy: {accuracy:.3f}")
    print(f"   - AUC-ROC:  {auc:.3f}")
    print(f"   - Class distribution: {(y_test == 0).sum()} favorable, {(y_test == 1).sum()} adverse")
    
    # ===== e) SENSITIVE ATTRIBUTES PRESENT =====
    print(f"\ne) Sensitive/Protected Attributes (from prep notebooks):")
    for attr in dataset_sensitive_attrs:
        print(f"   - {attr}")
    
    # ===== e2) SENSITIVE ATTRIBUTES IN ADVERSE INSTANCES =====
    available_attrs = [attr for attr in dataset_sensitive_attrs if attr in adverse_df.columns]
    print(f"\ne2) Sensitive Attributes in Adversely Predicted Instances:")
    
    if available_attrs:
        for attr in available_attrs:
            # Count how many adverse instances have this sensitive attribute (non-null)
            has_attr = adverse_df[attr].notna().sum()
            pct = (has_attr / len(adverse_df)) * 100
            print(f"   - {attr}: {has_attr}/{len(adverse_df)} instances ({pct:.1f}%)")
    else:
        print(f"   No sensitive attributes found in adverse instance data.")
    
    # ===== e3) DISTRIBUTION OF PROTECTED ATTRIBUTES =====
    print(f"\ne3) Distribution of Protected Attributes in Adverse Instances:")
    
    if available_attrs:
        for attr in available_attrs:
            print(f"\n   {attr} - Value Distribution:")
            value_counts = adverse_df[attr].value_counts().sort_index()
            for value, count in value_counts.items():
                pct = (count / len(adverse_df)) * 100
                print(f"      Value {value}: {count} instances ({pct:.1f}%)")
    else:
        print(f"   No protected attributes found in adverse instance data.")
    
    # ===== f) SUMMARY STATISTICS OF ADVERSE INSTANCES =====
    print(f"\nf) Summary Statistics of Adverse Instances:")
    
    if available_attrs:
        numeric_attrs = adverse_df[available_attrs].select_dtypes(include=[np.number]).columns.tolist()
        if numeric_attrs:
            print(f"\n   Numeric Attributes:")
            print(adverse_df[numeric_attrs].describe().to_string())
        
        categorical_attrs = adverse_df[available_attrs].select_dtypes(exclude=[np.number]).columns.tolist()
        if categorical_attrs:
            print(f"\n   Categorical Attributes Value Counts:")
            for attr in categorical_attrs:
                print(f"\n   {attr}:")
                print(adverse_df[attr].value_counts().to_string())
    else:
        print(f"   No sensitive attributes found in adverse_df.")
    
    missing_count = adverse_df[available_attrs].isnull().sum().sum() if available_attrs else 0
    print(f"\n   Missing values: {missing_count}")
    print(f"\n{'='*80}\n")

## Credit Dataset Analysis

In [54]:
# Credit Dataset - using exact protected_attributes from prep_credit.ipynb
credit_sensitive_attrs = ['age', 'personal_status_sex', 'foreign_worker']

analyze_dataset(
    dataset_name='credit',
    target_col='credit_risk',
    adverse_file='credit_adverse.csv',
    dataset_sensitive_attrs=credit_sensitive_attrs
)


DATASET: CREDIT

a) Total Instances in Dataset:
   - Train set: 800 instances
   - Test set:  200 instances
   - Total:     1,000 instances

b) Adversely Predicted Instances (class 1):
   - Total adverse in test set: 48 (24.0%)

c) Train/Test Split Used:
   - Train: 80.0%
   - Test:  20.0%

d) Model Performance on Test Set:
   - Accuracy: 0.735
   - AUC-ROC:  0.783
   - Class distribution: 130 favorable, 70 adverse

e) Sensitive/Protected Attributes (from prep notebooks):
   - age
   - personal_status_sex
   - foreign_worker

e2) Sensitive Attributes in Adversely Predicted Instances:
   - age: 48/48 instances (100.0%)
   - personal_status_sex: 48/48 instances (100.0%)
   - foreign_worker: 48/48 instances (100.0%)

e3) Distribution of Protected Attributes in Adverse Instances:

   age - Value Distribution:
      Value 20: 1 instances (2.1%)
      Value 22: 2 instances (4.2%)
      Value 23: 2 instances (4.2%)
      Value 24: 3 instances (6.2%)
      Value 25: 2 instances (4.2%)
      V

## Law Dataset Analysis

In [55]:
# Law Dataset - using exact protected_attributes from prep_law.ipynb
law_sensitive_attrs = ['gender', 'race1']

analyze_dataset(
    dataset_name='law',
    target_col='bar',
    adverse_file='law_adverse.csv',
    dataset_sensitive_attrs=law_sensitive_attrs
)


DATASET: LAW

a) Total Instances in Dataset:
   - Train set: 16,341 instances
   - Test set:  4,086 instances
   - Total:     20,427 instances

b) Adversely Predicted Instances (class 1):
   - Total adverse in test set: 349 (8.5%)

c) Train/Test Split Used:
   - Train: 80.0%
   - Test:  20.0%

d) Model Performance on Test Set:
   - Accuracy: 0.888
   - AUC-ROC:  0.834
   - Class distribution: 3652 favorable, 434 adverse

e) Sensitive/Protected Attributes (from prep notebooks):
   - gender
   - race1

e2) Sensitive Attributes in Adversely Predicted Instances:
   - gender: 349/349 instances (100.0%)
   - race1: 349/349 instances (100.0%)

e3) Distribution of Protected Attributes in Adverse Instances:

   gender - Value Distribution:
      Value 0: 178 instances (51.0%)
      Value 1: 171 instances (49.0%)

   race1 - Value Distribution:
      Value 0: 176 instances (50.4%)
      Value 1: 40 instances (11.5%)
      Value 2: 18 instances (5.2%)
      Value 3: 99 instances (28.4%)
      Va

## Saudi Dataset Analysis

In [56]:
# Saudi Dataset - using exact protected_attributes from prep_saudi.ipynb
saudi_sensitive_attrs = ['Gender', 'Age']

analyze_dataset(
    dataset_name='saudi',
    target_col='Attrition',
    adverse_file='saudi_adverse.csv',
    dataset_sensitive_attrs=saudi_sensitive_attrs
)


DATASET: SAUDI

a) Total Instances in Dataset:
   - Train set: 939 instances
   - Test set:  235 instances
   - Total:     1,174 instances

b) Adversely Predicted Instances (class 1):
   - Total adverse in test set: 112 (47.7%)

c) Train/Test Split Used:
   - Train: 80.0%
   - Test:  20.0%

d) Model Performance on Test Set:
   - Accuracy: 0.821
   - AUC-ROC:  0.882
   - Class distribution: 134 favorable, 101 adverse

e) Sensitive/Protected Attributes (from prep notebooks):
   - Gender
   - Age

e2) Sensitive Attributes in Adversely Predicted Instances:
   - Gender: 112/112 instances (100.0%)
   - Age: 112/112 instances (100.0%)

e3) Distribution of Protected Attributes in Adverse Instances:

   Gender - Value Distribution:
      Value 0: 62 instances (55.4%)
      Value 1: 50 instances (44.6%)

   Age - Value Distribution:
      Value 0: 53 instances (47.3%)
      Value 1: 50 instances (44.6%)
      Value 2: 9 instances (8.0%)

f) Summary Statistics of Adverse Instances:

   Numeric A

## Student Dataset Analysis

In [57]:
# Student Dataset - using exact protected_attributes from prep_student.ipynb
student_sensitive_attrs = ['sex', 'age', 'health', 'Pstatus']

analyze_dataset(
    dataset_name='student',
    target_col='target',
    adverse_file='student_adverse.csv',
    dataset_sensitive_attrs=student_sensitive_attrs
)


DATASET: STUDENT

a) Total Instances in Dataset:
   - Train set: 519 instances
   - Test set:  130 instances
   - Total:     649 instances

b) Adversely Predicted Instances (class 1):
   - Total adverse in test set: 5 (3.8%)

c) Train/Test Split Used:
   - Train: 80.0%
   - Test:  20.0%

d) Model Performance on Test Set:
   - Accuracy: 0.854
   - AUC-ROC:  0.828
   - Class distribution: 111 favorable, 19 adverse

e) Sensitive/Protected Attributes (from prep notebooks):
   - sex
   - age
   - health
   - Pstatus

e2) Sensitive Attributes in Adversely Predicted Instances:
   - sex: 5/5 instances (100.0%)
   - age: 5/5 instances (100.0%)
   - health: 5/5 instances (100.0%)
   - Pstatus: 5/5 instances (100.0%)

e3) Distribution of Protected Attributes in Adverse Instances:

   sex - Value Distribution:
      Value 0: 2 instances (40.0%)
      Value 1: 3 instances (60.0%)

   age - Value Distribution:
      Value 16: 1 instances (20.0%)
      Value 17: 2 instances (40.0%)
      Value 18: 2